In [135]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage,SystemMessage,HumanMessage
from typing import TypedDict, List,Optional, Annotated,Sequence

In [136]:
class Clarification(TypedDict):
    needs_improvement :  bool
    questions : Optional[List[str]]

class KeywordExtractionOutput(TypedDict):
    main_keywords: List[str]
    arxiv_phrases: List[str]
    semantic_scholar_queries: List[str]


class AgentState(TypedDict):
    clarification : Clarification
    keywords : KeywordExtractionOutput
    messages : Annotated[Sequence[BaseMessage],add_messages]

In [137]:
clarifier_llm = ChatGroq(model="moonshotai/kimi-k2-instruct").with_structured_output(Clarification)
keyword_llm = ChatGroq(model="llama3-70b-8192").with_structured_output(KeywordExtractionOutput)

In [138]:
def clarifier(state:AgentState):
    system_prompt = SystemMessage(content="""
        You are a research assistant. The user will provide you with a research paper topic or description.
        
        Your job is to check if the description includes:
        1. Research domain
        2. Problem being solved
        3. Method or technique used
        4. Any dataset mentioned
        
        If ANY of these elements are missing or unclear, set needs_improvement to True and provide specific questions in the 'questions' field to gather the missing information.
        
        If all elements are present and clear, set needs_improvement to False and questions can be null or empty.
        
        Example response format:
        - If missing info: {"needs_improvement": true, "questions": ["What specific problem are you trying to solve?", "Which dataset will you use?"]}
        - If complete: {"needs_improvement": false, "questions": null}
        """
    )
    clarifier_response = clarifier_llm.invoke([system_prompt]+[state["messages"][-1]])

    return {"clarification":clarifier_response}

In [139]:
def keyworder(state:AgentState):
    system_prompt = SystemMessage(content="""
        You are a research assistant trained to extract keywords for academic paper search engines, specifically for:
        - arXiv (https://arxiv.org)
        - Semantic Scholar (https://semanticscholar.org)

        You will receive a short research topic description from the user.

        Your task is to analyze the description and return a set of structured keywords and search phrases optimized for academic search.

        You MUST return your output as an instance of the following schema:

        class KeywordExtractionOutput(BaseModel):
            main_keywords: List[str]  # core technical terms, methods, datasets
            arxiv_phrases: List[str]  # short, concise phrases optimized for arXiv title/abstract search
            semantic_scholar_queries: List[str]  # natural language-style search strings for Semantic Scholar

        Guidelines:
        - Avoid generic terms like "paper", "study", "research"
        - Prefer specific methods (e.g., CNN, BERT, PCA), tasks (e.g., segmentation, prediction), and datasets (e.g., CHB-MIT, ImageNet)
        - If user input is vague, extract the most relevant, inferable terms — dont leave the lists empty
        - All outputs should be lowercase unless referring to acronyms (e.g., EEG, GNN, LSTM)

        Only return a valid Python object matching the schema exactly.
        Do not include any extra fields, strings, comments, or explanations.
        Avoid quoting the entire object as a string.
        """
        )
    
    keyworder_response = keyword_llm.invoke([system_prompt] + state['messages'])

    return {"keywords":keyworder_response}
    

In [140]:
def clarifier_router(state: AgentState):
    if state['clarification']['needs_improvement']:
        return "end"
    else:
        return "continue"

In [141]:
graph = StateGraph(AgentState)

graph.add_node("clarificationAgent",clarifier)
graph.add_node("keywordAgent",keyworder)

graph.add_edge(START,"clarificationAgent")

graph.add_conditional_edges(
    "clarificationAgent",
    clarifier_router,
    {
        "end":END,
        "continue": "keywordAgent"
    }
)

graph.add_edge("keywordAgent",END)

app = graph.compile()

In [144]:
user_input_messages = [HumanMessage(content="""
The title of my paper is Pix2pix++: an enhanced GAN based approach for image to image translation.
""")]

# Initialize state properly
state = {
    "messages": user_input_messages, 
    "clarification": {"needs_improvement": False, "question": None}, 
    "keywords": ""
}

for event in app.stream(state):
    for node_name, node_output in event.items():
        print(f"\n🧩 Agent: {node_name}")
        print(f"📦 Output: {node_output}")
        state.update(node_output)
        
    if node_name == END:
        break
        
    if state.get('clarification', {}).get('needs_improvement', False):
        message = input("Answer: ")
        if message.lower() == "exit":
            break
        state["messages"].append(HumanMessage(content=message))



🧩 Agent: clarificationAgent
📦 Output: {'clarification': {'needs_improvement': True, 'questions': ['What specific problem or limitation in existing image-to-image translation methods is Pix2pix++ trying to solve?', 'What dataset(s) will you use to train and evaluate Pix2pix++?', 'Can you briefly describe the key enhancements or new techniques introduced in Pix2pix++ compared to the original pix2pix?']}}


In [ ]:
# The title of my paper is preictal state recognition using geometric deep learning. Im trying to improve the early detection of preictal (pre-seizure) brain states in patients with epilepsy using EEG data. 
#                                     The goal is to predict seizure onset several minutes in advance so preventive interventions can be applied, especially in wearable or edge devices. 
#                                     I plan to use a Graph Neural Network (GNN) architecture, specifically a spatio-temporal GCN, to model both the spatial brain connectivity and the temporal patterns leading up to a seizure
#                                     .Ill be using the CHB-MIT Scalp EEG dataset, 
#                                     which contains long-term EEG recordings from pediatric subjects with intractable seizures, 
#                                     including annotations for seizure onset and preictal windows.